In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Change this to egl or glfw if available
os.environ["MUJOCO_GL"] = "egl"
import mediapy as media
import huggingface_hub as hf_hub


from track_mjx.agent import checkpointing
from track_mjx.analysis.utils import save_to_h5py
from vnl_mjx.tasks.celegans.imitation import Imitation
from vnl_mjx.tasks.celegans.reference_clips import ReferenceClips

import jax
import wandb
from jax import numpy as jp
from pathlib import Path
from omegaconf import OmegaConf


In [ ]:
def create_rollout_generator(
    cfg,
    environment,
    inference_fn,
    model="mlp",
    log_states=False,
    log_activations=False,
    log_metrics=False,
    log_sensor_data=False,
):
    """
    Creates a rollout generator with JIT-compiled functions.

    Args:
        environment (Env): The environment to generate rollouts for.
        inference_fn (Callable): The inference function to compute controls.

    Returns:
        Callable: A generate_rollout function that can be called with configuration.
    """
    ref_traj_config = cfg["reference_config"]
    # Wrap the environment
    # TODO this logic is used in a few different places, make it a function?
    rollout_env = environment  # Initialize with base environment

    # JIT-compile the necessary functions
    jit_inference_fn = jax.jit(inference_fn)
    jit_reset = jax.jit(rollout_env.reset)
    jit_step = jax.jit(rollout_env.step)

    def generate_rollout(clip_idx: int | None = None, seed: int = 42):
        """
        Generates a rollout using pre-compiled JIT functions.

        Args:
            clip_idx (Optional[int]): Specific clip ID to generate the rollout for.
            seed (int): Random seed for jax PRNGKey.
            log_activations (bool): Whether to log neural network activations.
            log_metrics (bool): Whether to log rollout metrics.
            log_sensor_data (bool): Whether to log sensor readings.

        Returns:
            Dict: A dictionary containing rollout data.
        """

        # Initialize PRNG keys
        rollout_key = jax.random.PRNGKey(seed)
        rollout_key, reset_rng, act_rng = jax.random.split(rollout_key, 3)

        # Reset the environment
        init_state = jit_reset(reset_rng, clip_idx=clip_idx, start_frame=0)

        num_steps = (
            int(ref_traj_config.clip_length * environment._steps_for_cur_frame) - 1
        )

        def _step_fn_mlp(carry, _):
            state, act_rng = carry
            act_rng, new_rng = jax.random.split(act_rng)
            ctrl, extras = jit_inference_fn(state.obs, act_rng)
            next_state = jit_step(state, ctrl)

            # Collect optional data based on logging flags
            joint_force = next_state.data.cfrc_ext if log_sensor_data else None
            sensor_reading = next_state.data.sensordata if log_sensor_data else None
            activations = extras["activations"] if log_activations else None

            return (next_state, new_rng), (
                next_state,
                ctrl,
                activations,
                joint_force,
                sensor_reading,
            )

        def _step_fn_lstm(carry, _):
            state, act_rng, hidden = carry
            act_rng, new_rng = jax.random.split(act_rng)
            ctrl, extras, new_hidden = jit_inference_fn(state.obs, act_rng, hidden)
            ctrl = jp.squeeze(ctrl, axis=0)
            next_state = jit_step(state, ctrl)

            # Collect optional data based on logging flags
            joint_force = next_state.data.cfrc_ext if log_sensor_data else None
            sensor_reading = next_state.data.sensordata if log_sensor_data else None
            activations = extras["activations"] if log_activations else None

            return (next_state, new_rng, new_hidden), (
                next_state,
                ctrl,
                hidden,
                activations,
                joint_force,
                sensor_reading,
            )

        # Initialize variables
        states = None
        ctrls = None
        activations = None
        joint_forces = None
        sensor_readings = None
        stacked_hidden = None

        if model == "mlp":
            # Run rollout for mlp
            init_carry = (init_state, jax.random.PRNGKey(0))
            (
                (final_state, _),
                (
                    states,
                    ctrls,
                    activations,
                    joint_forces,
                    sensor_readings,
                ),
            ) = jax.lax.scan(_step_fn_mlp, init_carry, None, length=num_steps)

        elif model == "lstm":
            # Run rollout for lstm
            init_carry = (
                init_state,
                jax.random.PRNGKey(0),
                init_state.info["hidden_state"],
            )
            (
                (final_state, _, final_hidden_state),
                (
                    states,
                    ctrls,
                    stacked_hidden,
                    activations,
                    joint_forces,
                    sensor_readings,
                ),
            ) = jax.lax.scan(_step_fn_lstm, init_carry, None, length=num_steps)

        def prepend(element, arr):
            # Scalar elements shouldn't be modified
            if arr.ndim == 0:
                return arr
            return jp.concatenate([element[None], arr])

        rollout_states = jax.tree.map(prepend, init_state, states)

        # Reference and rollout qposes (always logged)
        def _get_ref_qpos(state):
            time_in_frames = state.data.time * env._config.mocap_hz
            frame = jp.floor(time_in_frames + state.info["start_frame"]).astype(int)
            clip = state.info["reference_clip"]
            ref = env.reference_clips.at(clip=clip, frame=frame)
            return ref.qpos

        qposes_ref = jax.vmap(_get_ref_qpos)(rollout_states)

        # Collect qposes from states (always logged)
        qposes_rollout = jax.vmap(lambda s: s.data.qpos)(rollout_states)

        # Extract state rewards (always logged)
        state_rewards = jax.vmap(lambda s: s.reward)(rollout_states)

        # Build return dictionary with required data
        result = {
            "qposes_ref": qposes_ref,
            "qposes_rollout": qposes_rollout,
            "ctrl": ctrls,
            "state_rewards": state_rewards,
        }

        # Add optional data if requested
        if log_metrics:
            rollout_metrics = {}
            for rollout_metric in cfg.logging_config.rollout_metrics:
                rollout_metrics[f"{rollout_metric}s"] = jax.vmap(
                    lambda s: s.metrics[rollout_metric]
                )(rollout_states)
            result["rollout_metrics"] = rollout_metrics

        if log_activations and activations is not None:
            result["activations"] = activations

        if log_sensor_data:
            if joint_forces is not None:
                result["joint_forces"] = joint_forces
            if sensor_readings is not None:
                result["sensor_readings"] = sensor_readings

        if log_states:

            def split_state_jit(batched_state: State):
                """
                JIT-compiled version that splits a batched State into individual States.
                This version uses jax.vmap for better performance.

                Args:
                    batched_state: A State object where each field has a leading batch dimension

                Returns:
                    A list of State objects, one for each element in the batch
                """
                # Get batch size
                batch_size = None

                def find_batch_size(x):
                    nonlocal batch_size
                    if isinstance(x, jax.Array) and x.ndim > 0:
                        if batch_size is None:
                            batch_size = x.shape[0]
                    return x

                jax.tree_util.tree_map(find_batch_size, batched_state)

                if batch_size is None:
                    raise ValueError("Could not determine batch size from State object")

                # Use jax.vmap to efficiently extract each element
                def extract_single_state(i):
                    def get_element_at_index(x):
                        if isinstance(x, jax.Array):
                            if x.ndim == 0:
                                return x
                            else:
                                return x[i]
                        else:
                            return x

                    return jax.tree_util.tree_map(get_element_at_index, batched_state)

                # Create indices for each batch element
                indices = jp.arange(batch_size)

                # Use vmap to extract all states at once
                states_array = jax.vmap(extract_single_state)(indices)

                # Convert to list
                return [
                    jax.tree_util.tree_map(lambda x: x[i], states_array)
                    for i in range(batch_size)
                ]

            result["states"] = split_state_jit(rollout_states)

        return result

    return jax.jit(generate_rollout)


In [ ]:
hf_checkpoint_path = "worm"
model_local_dir = Path.cwd().parent / "model_checkpoints"
# Download model from model repo
model_download_dir = hf_hub.snapshot_download(
    repo_id="talmolab/MIMIC-MJX",
    repo_type="model",  # download from model repo
    allow_patterns=hf_checkpoint_path + "/*",  # path with model id
    local_dir=model_local_dir,
)
print(f"Downloaded model to {model_download_dir}")


In [ ]:
hf_data_path = "data/worm/celegans_ik_only_04182019am_centerline_locomotion_2d.h5"
# Download data from dataset repo
data_download_dir = hf_hub.hf_hub_download(
    repo_id="talmolab/MIMIC-MJX",
    repo_type="dataset",  # download from dataset repo
    filename=hf_data_path,  # dataset name
    local_dir=Path.cwd().parent,
)
print(f"Downloaded data to {data_download_dir}")


In [ ]:
# replace with your checkpoint path
ckpt_path = model_local_dir / hf_checkpoint_path
# Load config from checkpoint
ckpt = checkpointing.load_checkpoint_for_eval(ckpt_path)

cfg = ckpt["cfg"]

# make some changes to the config
# replace with absolute path to your data
# -- your notebook may not have access to the same relative path
# cfg.data_path = Path.cwd().parent / "data/transform_snips.h5"
# cfg.reference_config = {"clip_length": 250, "random_init_range": 50, "traj_length": 5}
cfg.train_setup.checkpoint_to_restore = ckpt_path
cfg.keys()


In [ ]:
env_args = cfg.env_config.env_args
env_args = OmegaConf.to_container(env_args)
# env_args.pop("reference_data_path")
# env_args.pop("clip_length")
env_args


In [ ]:
inference_fn = checkpointing.load_inference_fn(cfg, ckpt["policy"])
data_path = data_download_dir
env = Imitation(config_overrides=env_args)
env.reference_clips = ReferenceClips(
    data_path,
    n_frames_per_clip=250,
)

generate_rollout = create_rollout_generator(
    cfg,
    env,
    inference_fn,
    log_states=False,
    log_activations=False,
    log_metrics=True,
    log_sensor_data=False,
)

In [ ]:
single_rollout = generate_rollout(
    clip_idx=0,
)
single_rollout[0]


In [ ]:
generate_rollout = create_rollout_generator(
    cfg,
    env,
    inference_fn,
    log_states=False,
    log_activations=False,
    log_metrics=True,
    log_sensor_data=False,
)
jit_vmap_generate_rollout = jax.jit(jax.vmap(generate_rollout))
clip_idxs = jp.arange(len(env.reference_clips))
jit_vmap_out = jit_vmap_generate_rollout(clip_idxs)


In [ ]:
save_to_h5py(
    f"{data_download_dir.split('/')[-1].split('.')[0]}_rollout.h5", jit_vmap_out
)